In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# BERT
- Encoder-only model
- Bidirectional attention
- Usage: Understanding, classification, extraction
- Mask: padding mask only -> every token can attend to all valid tokens
## Core idea
- Topo: X (B, n, d) -> Encoder -> H (B, n, d)
- Bidirectional (no causal masking)
- Designed to understand representations, not natural left-to-right generation
## Training
### Masked Language Modeling (MLM)
Randomly mask some tokens and predict them from bidirectional context: p(x_masked | x_visible)
- Force the model to build contextual representation using both left and right context
### Next Sentence Prediction (NSP)
Original BERT training, removed by later models/modified
## Output
- BERT is a contextual feature extractor: each token vector depends on the full valid sequence
### Token-level representation
H (B, n, d)
- Used in NER, token classification, extractive QA
### Sequence-level representation
[CLS] token representation
h_CLS (B, d)
- Used in classification/sentence-pair task
### Usage
- Text classification 
- Named Entity Recognization (NER)
- Extractive Question Answering
- Sentence-pair classification
- Embedding extraction for retrieval/ranking pipelines
- NOT: long free-form generation

In [6]:
ckpt = "bert-base-uncased"
tok = AutoTokenizer.from_pretrained(ckpt)
model = AutoModelForSequenceClassification.from_pretrained(
    ckpt,
    num_labels=3,
)
batch = tok(
    ["This movie is excellent!", "This movie is bad.", "This movie is okay."],
    padding=True,
    truncation=True,
    return_tensors="pt",
)
out = model(**batch)
print(torch.softmax(out.logits, dim=-1))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9311.32it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

tensor([[0.3547, 0.4212, 0.2241],
        [0.3581, 0.4611, 0.1808],
        [0.3531, 0.4563, 0.1906]], grad_fn=<SoftmaxBackward0>)


## Common pitfalls
- Wrong tokenizer for ckpt
- Forget padding mask
- Treat BERT as left-to-right generator
- [CLS] blindly used for semantic similarity without checking embedding quality
- Truncating long documents without a chunking strategy
- For document-level tasks, BERT's max length forces chunking, pooling, long-context variants